# 🟢 2.2 Architectural Planning (`Shift+Tab`): Mastering `/plan` Mode Before Execution
**Course:** Agentic CLI Engineering - Complete Claude Code Masterclass  
**Module:** Section 2 — Interactive Session Mastery & Context Hygiene  
**Shared Section 2 Project:** `./order_service` (`v2.1-baseline` $\rightarrow$ `v2.2-planned-and-verified`)  
**Configured OpenRouter Model:** `deepseek/deepseek-v4.1-flash-1017`  

---

## 🏛️ Architectural Objective
In **Lecture 2.1**, our Read-First Reconnaissance of `./order_service` uncovered two critical defects:
1. **A `2N + 1` SQL Query Explosion (`11` queries for `5` customers)** in `services/report_builder.py` and `db/repository.py`.
2. **An Unverified JWT & `X-Internal-Debug` Header Backdoor** in `auth/jwt_guard.py`.

If you immediately prompt an AI to *"Rewrite all the files to fix this,"* it will make silent assumptions about Redis key expirations, cache-miss fallbacks, and token rotation state—leading to broken multi-file diffs.

Instead, we use **Plan Mode** (`Shift+Tab` twice or `/plan` interactively in the **Claude CLI Terminal**, or `--permission-mode plan`). In this hands-on lab, you will:
1. **Verify the Shared `./order_service` Baseline** from Lecture 2.1.
2. **Execute `/plan` Mode in Your Claude CLI Terminal** (`--model "deepseek/deepseek-v4.1-flash-1017" --permission-mode plan`) to generate a formal, read-only `ARCHITECTURE_PLAN.md` specification.
3. **Stress-Test & Execute the Approved Blueprint** (Batch SQL `IN (...)` queries + Redis Cache-Aside with fail-open fallback + HMAC-SHA256 JWT Refresh-Token Reuse Detection).
4. **Verify Convergence via `pytest`** (`11 SQL queries -> 3 on Cold Cache -> 0 on Warm Redis Hit`) and lock Git milestone `v2.2-planned-and-verified`.

---
## Phase 1: Verifying the Shared `./order_service` Baseline
Run the cell below to verify that `./order_service` is staged at `v2.1-baseline` with `.claude/settings.json` bound to `deepseek/deepseek-v4.1-flash-1017` (includes a self-healing guard if you opened Lecture 2.2 directly).

In [ ]:
%%bash
# 0. Auto-sync required packages from root requirements.txt
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")
REQ_FILE=$( [ -f "requirements.txt" ] && echo "requirements.txt" || echo "../requirements.txt" )
if [ -f "$REQ_FILE" ]; then
    echo -e "\033[0;36m[SETUP] Syncing Python dependencies from $REQ_FILE...\033[0m"
    $PY_BIN -m pip install -q -r "$REQ_FILE"
fi

# Self-healing guard: initialize ./order_service if Lecture 2.1 wasn't run yet
if [ ! -d "order_service/.git" ]; then
    mkdir -p order_service/routes order_service/auth order_service/services order_service/db order_service/utils order_service/.claude
    cd order_service
    printf "__pycache__/\n*.pyc\n.pytest_cache/\n.env*\n.claude/\n" > .gitignore
    touch routes/__init__.py auth/__init__.py services/__init__.py db/__init__.py utils/__init__.py

    cat << 'EOF' > db/repository.py
class LegacySQLClient:
    def __init__(self):
        self.query_count = 0

    def fetch_active_customer_ids(self, region: str) -> list[int]:
        self.query_count += 1
        return [101, 102, 103, 104, 105]

    def fetch_orders_for_customer(self, customer_id: int) -> list[dict]:
        self.query_count += 1
        return [{"order_id": f"ORD-{customer_id}-A", "customer_id": customer_id, "total_cents": 4500}]

    def fetch_refunds_for_customer(self, customer_id: int) -> list[dict]:
        self.query_count += 1
        return [{"refund_id": f"REF-{customer_id}", "customer_id": customer_id, "refund_cents": 500}]
EOF

    cat << 'EOF' > auth/jwt_guard.py
JWT_OPTIONS = {"verify_signature": False, "verify_exp": False}

def authenticate_request(headers: dict) -> dict:
    if headers.get("X-Internal-Debug") == "true":
        return {"user_id": "system_override", "role": "ADMIN", "verified": False}
    return {"user_id": "unverified", "role": "ADMIN", "verified": False}
EOF

    cat << 'EOF' > utils/currency.py
def cents_to_usd(cents: int) -> str:
    return f"${cents / 100:.2f}"
EOF

    cat << 'EOF' > services/report_builder.py
from db.repository import LegacySQLClient
from utils.currency import cents_to_usd

def build_regional_settlement_report(db: LegacySQLClient, region: str) -> dict:
    customer_ids = db.fetch_active_customer_ids(region)
    report_rows = []
    for cid in customer_ids:
        orders = db.fetch_orders_for_customer(cid)
        refunds = db.fetch_refunds_for_customer(cid)
        gross = sum(o["total_cents"] for o in orders)
        refunded = sum(r["refund_cents"] for r in refunds)
        report_rows.append({"customer_id": cid, "net_revenue": cents_to_usd(gross - refunded)})
    return {"region": region, "total_sql_queries_executed": db.query_count, "data": report_rows}
EOF

    git init -q
    git config user.email "architect@fnosystematics.co.in"
    git config user.name "Agentic Architect"
    git add . && git commit -q -m "chore(init): baseline legacy order_service for Section 2"
    git tag -f v2.1-baseline
    cd ..
fi

cd order_service
mkdir -p .claude
SAMPLE_SETTINGS="../../Foundation & Memory Architecture/lab_1_2_workspace/.claude_sample/settings.json"
if [ -f "$SAMPLE_SETTINGS" ]; then
    cp "$SAMPLE_SETTINGS" .claude/settings.json
fi

echo -e "\033[1;32m[✔] Shared Section 2 repository ready in ./order_service (HEAD: $(git rev-parse --short HEAD))\033[0m"
echo -e "\033[1;32m[✔] Active OpenRouter Model: deepseek/deepseek-v4.1-flash-1017\033[0m"

---
## Phase 2: 🖥️ Execute `/plan` Mode in Your Claude CLI Terminal
Open your **Claude CLI Terminal** inside `./order_service` and run **Plan Mode** (`Shift+Tab` twice interactively, or `--permission-mode plan`) using **`deepseek/deepseek-v4.1-flash-1017`**.

### Option A: Interactive `/plan` Mode in Your Terminal (Recommended for Video Demo)
```bash
# 1. Enter the shared Section 2 project directory
cd "/Users/anshul/Training/Agentic_CLI_Engineering/Interactive Session Mastery & Context Hygiene/order_service"

# 2. Launch Claude Code with deepseek/deepseek-v4.1-flash-1017
claude --model "deepseek/deepseek-v4.1-flash-1017"

# 3. Inside Claude Code, press Shift+Tab twice (or type /plan) to enter Plan Mode:
> /plan
> Inspect this repository and design a concise markdown architectural specification covering: (1) Refactoring db/repository.py and services/report_builder.py from 2N+1 SQL queries to O(1) batch IN queries, (2) Adding a Redis Cache-Aside layer (services/redis_cache.py) with graceful fail-open fallback if Redis is down, and (3) Upgrading auth/jwt_guard.py to HMAC-SHA256 verification with Refresh-Token Family Rotation and stolen token reuse detection. Do NOT modify any code files.
```

### Option B: One-Shot CLI `--permission-mode plan` in Your Terminal
```bash
cd "/Users/anshul/Training/Agentic_CLI_Engineering/Interactive Session Mastery & Context Hygiene/order_service"

claude -p "You are in /plan mode. Inspect this repository and output a concise markdown architectural specification covering: (1) Refactoring db/repository.py and services/report_builder.py from 2N+1 SQL queries to O(1) batch IN queries, (2) Adding a Redis Cache-Aside layer (services/redis_cache.py) with graceful fallback if Redis is down, and (3) Upgrading auth/jwt_guard.py to HMAC-SHA256 verification with Refresh-Token Family Rotation and stolen token reuse detection. Do NOT modify any code files." \
  --model "deepseek/deepseek-v4.1-flash-1017" \
  --permission-mode plan \
  --allowedTools "Read,Grep,Glob" | tee ARCHITECTURE_PLAN.md
```

Run the cell below to ensure `ARCHITECTURE_PLAN.md` is locked in `./order_service/ARCHITECTURE_PLAN.md` and verify that **zero tracked code files were modified during Plan Mode**.

In [ ]:
%%bash
cd order_service

if [ ! -s "ARCHITECTURE_PLAN.md" ]; then
    cat << 'EOF' > ARCHITECTURE_PLAN.md
# ARCHITECTURAL SPECIFICATION (GENERATED IN `/plan` MODE)
**Model:** `deepseek/deepseek-v4.1-flash-1017`

## 1. Database Layer: O(1) Batch Query Refactor (`db/repository.py`)
- **Problem:** `fetch_orders_for_customer` and `fetch_refunds_for_customer` execute inside an N-iteration loop (`2N + 1` queries).
- **Design:** Add `fetch_orders_batch(customer_ids: list[int])` and `fetch_refunds_batch(customer_ids: list[int])` simulating `WHERE customer_id IN (...)`.
- **Complexity Target:** Strictly `3` SQL queries on a cold cache regardless of customer count `N`.

## 2. Caching Layer: Redis Cache-Aside with Circuit-Breaker (`services/redis_cache.py`)
- **Key Namespace:** `settlement:report:v1:{region}` with `TTL = 300s`.
- **Edge Case (Redis Down):** Wrap `get()` and `setex()` in a fail-open check so if Redis is offline, the service falls back seamlessly to the Batch SQL path without failing the HTTP request.

## 3. Security Layer: HMAC-SHA256 JWT & Refresh-Token Family Rotation (`auth/jwt_guard.py`)
- **Remove Backdoor:** Delete the `X-Internal-Debug` header bypass immediately.
- **Cryptographic Signature:** Verify `HMAC-SHA256` signature on all bearer tokens.
- **Refresh-Token Reuse Detection:** Track token families (`family_id`). When a refresh token is rotated, mark its `jti` as consumed. If a consumed `jti` is ever presented again (stolen token replay), immediately revoke the entire `family_id`.
EOF
fi

cat ARCHITECTURE_PLAN.md
echo ""
echo -e "\033[0;32m[✔] Spec locked in ARCHITECTURE_PLAN.md | Tracked git files modified during Plan Mode: $(git diff --name-only | wc -l | tr -d ' ')\033[0m"

---
## Phase 3: Executing the Approved Blueprint
Look at `ARCHITECTURE_PLAN.md` above: before a single line of Python was changed, we locked down:
* **Exact SQL query complexity** (`2N+1` $\rightarrow$ `3` batch queries).
* **Exact Redis failure behavior** (fail-open fallback if Redis goes down).
* **Exact Token Replay defense** (consuming `jti` and revoking the entire `family_id` on replay).

Now we exit `/plan` mode and execute the approved blueprint across all 4 modules inside `./order_service`.

In [ ]:
%%writefile order_service/db/repository.py
# ✔ STEP 1 OF PLAN: O(1) Batch SQL Query Methods
from collections import defaultdict

class LegacySQLClient:
    def __init__(self):
        self.query_count = 0

    def fetch_active_customer_ids(self, region: str) -> list[int]:
        self.query_count += 1
        return [101, 102, 103, 104, 105]

    def fetch_orders_batch(self, customer_ids: list[int]) -> dict[int, list[dict]]:
        """Single batch SQL query: SELECT * FROM orders WHERE customer_id IN (...) """
        if not customer_ids:
            return {}
        self.query_count += 1
        grouped = defaultdict(list)
        for cid in customer_ids:
            grouped[cid].append({"order_id": f"ORD-{cid}-A", "customer_id": cid, "total_cents": 4500})
        return dict(grouped)

    def fetch_refunds_batch(self, customer_ids: list[int]) -> dict[int, list[dict]]:
        """Single batch SQL query: SELECT * FROM refunds WHERE customer_id IN (...) """
        if not customer_ids:
            return {}
        self.query_count += 1
        grouped = defaultdict(list)
        for cid in customer_ids:
            grouped[cid].append({"refund_id": f"REF-{cid}", "customer_id": cid, "refund_cents": 500})
        return dict(grouped)


In [ ]:
%%writefile order_service/services/redis_cache.py
# ✔ STEP 2 OF PLAN: Redis Cache-Aside Client with Fail-Open Circuit Breaker
import json
from typing import Optional, Dict, Any

class RedisCacheClient:
    def __init__(self, online: bool = True):
        self.online = online
        self._store: Dict[str, str] = {}

    def get_json(self, key: str) -> Optional[Dict[str, Any]]:
        if not self.online:
            # Fail-open fallback when Redis cluster is unreachable
            return None
        raw = self._store.get(key)
        return json.loads(raw) if raw else None

    def setex_json(self, key: str, ttl_seconds: int, payload: Dict[str, Any]) -> None:
        if not self.online:
            return
        self._store[key] = json.dumps(payload)


In [ ]:
%%writefile order_service/services/report_builder.py
# ✔ STEP 3 OF PLAN: Batch Aggregation + Redis Cache-Aside Integration
from typing import Optional
from db.repository import LegacySQLClient
from services.redis_cache import RedisCacheClient
from utils.currency import cents_to_usd

CACHE_TTL_SECONDS = 300

def build_regional_settlement_report(
    db: LegacySQLClient,
    region: str,
    cache: Optional[RedisCacheClient] = None
) -> dict:
    cache_key = f"settlement:report:v1:{region}"

    # 1. Check Redis Cache-Aside first
    if cache is not None:
        cached_report = cache.get_json(cache_key)
        if cached_report is not None:
            cached_report["cache_hit"] = True
            cached_report["total_sql_queries_executed"] = db.query_count
            return cached_report

    # 2. Cold Cache Path: Execute strictly 3 Batch SQL Queries (O(1) query count)
    customer_ids = db.fetch_active_customer_ids(region)
    orders_by_customer = db.fetch_orders_batch(customer_ids)
    refunds_by_customer = db.fetch_refunds_batch(customer_ids)

    report_rows = []
    for cid in customer_ids:
        orders = orders_by_customer.get(cid, [])
        refunds = refunds_by_customer.get(cid, [])
        gross = sum(o["total_cents"] for o in orders)
        refunded = sum(r["refund_cents"] for r in refunds)
        report_rows.append({
            "customer_id": cid,
            "net_revenue": cents_to_usd(gross - refunded)
        })

    result = {
        "region": region,
        "customers_processed": len(customer_ids),
        "total_sql_queries_executed": db.query_count,
        "cache_hit": False,
        "data": report_rows
    }

    # 3. Populate Redis Cache with 300s TTL
    if cache is not None:
        cache.setex_json(cache_key, CACHE_TTL_SECONDS, result)

    return result


In [ ]:
%%writefile order_service/auth/jwt_guard.py
# ✔ STEP 4 OF PLAN: HMAC-SHA256 Verification + Refresh-Token Family Rotation
import hashlib
import hmac
import json
import uuid

SECRET_KEY = b"production-256-bit-hmac-signing-secret"

def sign_token(payload: dict) -> str:
    body = json.dumps(payload, sort_keys=True).encode("utf-8").hex()
    sig = hmac.new(SECRET_KEY, body.encode("utf-8"), hashlib.sha256).hexdigest()
    return f"{body}.{sig}"

def authenticate_request(headers: dict) -> dict:
    # X-Internal-Debug backdoor permanently removed per ARCHITECTURE_PLAN.md
    auth_header = headers.get("Authorization", "")
    if not auth_header.startswith("Bearer "):
        raise PermissionError("Missing or malformed Bearer token")

    token = auth_header.replace("Bearer ", "", 1)
    try:
        body_hex, provided_sig = token.split(".")
    except ValueError:
        raise PermissionError("Invalid token structure")

    expected_sig = hmac.new(SECRET_KEY, body_hex.encode("utf-8"), hashlib.sha256).hexdigest()
    if not hmac.compare_digest(provided_sig, expected_sig):
        raise PermissionError("HMAC-SHA256 signature verification failed!")

    payload = json.loads(bytes.fromhex(body_hex).decode("utf-8"))
    return {"user_id": payload["sub"], "role": payload["role"], "verified": True}

class RefreshTokenFamilyStore:
    """Implements Refresh-Token Rotation with Stolen Token Reuse Detection."""
    def __init__(self):
        self.active_jti_by_family: dict[str, str] = {}
        self.consumed_jtis: dict[str, str] = {}  # jti -> family_id
        self.revoked_families: set[str] = set()

    def issue_family(self, user_id: str) -> tuple[str, str]:
        family_id = f"fam-{user_id}-{uuid.uuid4().hex[:6]}"
        jti = f"jti-{uuid.uuid4().hex[:8]}"
        self.active_jti_by_family[family_id] = jti
        return family_id, jti

    def rotate_refresh_token(self, family_id: str, presented_jti: str) -> str:
        if family_id in self.revoked_families:
            raise PermissionError("Token family is revoked! Re-authentication required.")

        # REUSE DETECTION: If a previously consumed JTI is presented again, a theft occurred!
        if presented_jti in self.consumed_jtis:
            self.revoked_families.add(family_id)
            raise PermissionError("Stolen refresh token reuse detected! Entire token family revoked.")

        if self.active_jti_by_family.get(family_id) != presented_jti:
            raise PermissionError("Unknown refresh token JTI.")

        # Consume current JTI and issue rotated child JTI
        self.consumed_jtis[presented_jti] = family_id
        new_jti = f"jti-{uuid.uuid4().hex[:8]}"
        self.active_jti_by_family[family_id] = new_jti
        return new_jti


---
## Phase 4: Verifying the Planned Architecture via `pytest` & Locking `v2.2-planned-and-verified`
Now let's write and run `test_planned_architecture.py` inside `./order_service` to verify that every requirement from our `/plan` mode specification passes deterministically, then commit milestone `v2.2-planned-and-verified` for **Lecture 2.3**.

In [ ]:
%%writefile order_service/test_planned_architecture.py
import pytest
from db.repository import LegacySQLClient
from services.redis_cache import RedisCacheClient
from services.report_builder import build_regional_settlement_report
from auth.jwt_guard import sign_token, authenticate_request, RefreshTokenFamilyStore

def test_batch_sql_and_redis_cache_hit():
    db = LegacySQLClient()
    cache = RedisCacheClient(online=True)

    # 1. Cold Cache: Should execute strictly 3 batch SQL queries (down from 11 in legacy 2N+1)
    cold_res = build_regional_settlement_report(db, "APAC", cache=cache)
    assert cold_res["cache_hit"] is False
    assert cold_res["total_sql_queries_executed"] == 3

    # 2. Warm Cache: Reset DB counter; Redis hit should execute 0 SQL queries!
    db.query_count = 0
    warm_res = build_regional_settlement_report(db, "APAC", cache=cache)
    assert warm_res["cache_hit"] is True
    assert warm_res["total_sql_queries_executed"] == 0

def test_redis_outage_fail_open_fallback():
    # Edge Case from Plan Mode: If Redis is offline, fallback cleanly to Batch SQL
    db = LegacySQLClient()
    dead_cache = RedisCacheClient(online=False)
    res = build_regional_settlement_report(db, "APAC", cache=dead_cache)
    assert res["cache_hit"] is False
    assert res["total_sql_queries_executed"] == 3
    assert len(res["data"]) == 5

def test_debug_backdoor_removed_and_hmac_enforced():
    # Legacy X-Internal-Debug bypass must fail
    with pytest.raises(PermissionError):
        authenticate_request({"X-Internal-Debug": "true"})

    # Tampered signature must fail
    valid_token = sign_token({"sub": "usr_1", "role": "ADMIN"})
    tampered_token = valid_token[:-4] + "0000"
    with pytest.raises(PermissionError):
        authenticate_request({"Authorization": f"Bearer {tampered_token}"})

    # Valid HMAC token succeeds
    principal = authenticate_request({"Authorization": f"Bearer {valid_token}"})
    assert principal["verified"] is True
    assert principal["role"] == "ADMIN"

def test_refresh_token_replay_revokes_entire_family():
    store = RefreshTokenFamilyStore()
    family_id, jti_1 = store.issue_family("usr_99")

    # Legitimate rotation: jti_1 -> jti_2
    jti_2 = store.rotate_refresh_token(family_id, jti_1)

    # Attacker replays stolen jti_1 -> Must trigger family revocation!
    with pytest.raises(PermissionError, match="Stolen refresh token reuse detected"):
        store.rotate_refresh_token(family_id, jti_1)

    # Even jti_2 is now blocked because the entire family was compromised and revoked
    with pytest.raises(PermissionError, match="Token family is revoked"):
        store.rotate_refresh_token(family_id, jti_2)


In [ ]:
%%bash
cd order_service
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")

echo -e "\033[1;32m=========================================================\033[0m"
echo -e "\033[1;32m [PHASE 4] VERIFYING '/plan' MODE IMPLEMENTATION         \033[0m"
echo -e "\033[1;32m=========================================================\033[0m"
$PY_BIN -m pytest test_planned_architecture.py -v

git add .
git commit -q -m "feat(plan): execute ARCHITECTURE_PLAN.md (batch SQL, Redis cache-aside, HMAC rotation)" || true
git tag -f v2.2-planned-and-verified

echo ""
echo -e "\033[1;32m[GIT MILESTONE LOCKED] Tag 'v2.2-planned-and-verified' on ./order_service\033[0m"
git log --oneline --decorate --graph

---
## 🖥️ Live Interactive Terminal Recording Guide (For Your Video Demo)
When demonstrating `/plan` mode live in your split-screen terminal:

```bash
# 1. Enter shared Section 2 workspace and launch interactive Claude Code
cd "/Users/anshul/Training/Agentic_CLI_Engineering/Interactive Session Mastery & Context Hygiene/order_service"
claude --model "deepseek/deepseek-v4.1-flash-1017"

# 2. Inside Claude Code, press Shift+Tab twice (or type /plan) to enter Plan Mode:
> /plan
> Design a plan to replace the 2N+1 SQL loop in report_builder.py with batch queries and Redis caching, and add HMAC + refresh token reuse detection to jwt_guard.py.

# 3. Or run non-interactively from bash:
claude -p "Create a refactor plan for batch SQL and Redis caching" \
  --model "deepseek/deepseek-v4.1-flash-1017" \
  --permission-mode plan
```

---
## 🎯 Architect's Wrap-Up
By separating **Architectural Planning (`/plan`)** from **Code Mutation** inside our shared `./order_service` project, we eliminated the `2N+1` SQL bottleneck (`11 queries -> 3 cold -> 0 warm`), survived a simulated Redis cluster outage, and blocked a stolen refresh-token replay attack on the very first implementation pass.

What happens when you are 45 minutes into a massive architectural session and your context window hits **85% capacity**—threatening to degrade model reasoning and burn huge token fees? In **Lecture 2.3: Context Window Hygiene: Token Telemetry (`/context`) and Surgical `/compact`**, we will master live context compression on `./order_service`.